# GDPR Compliance — Right to Erasure & PII Masking

This notebook demonstrates LakeLogic's GDPR capabilities:
1. **Right-to-be-Forgotten** — Erase PII for specific data subjects
2. **PII Masking** — Anonymise entire datasets for dev/test environments
3. **Audit Reports** — Generate compliance-ready erasure documentation

All driven by the contract's `pii: true` field annotations.

## Setup

In [ ]:
from pathlib import Path
import polars as pl

from lakelogic.core.processor import DataProcessor
from lakelogic.core.gdpr import forget_subjects, mask_pii_columns, generate_erasure_report

CONTRACT = Path("customer_contract.yaml")
DATA     = Path("data/customers.csv")

proc = DataProcessor(str(CONTRACT), engine="polars")
df   = pl.read_csv(str(DATA))
df

## 1. Right-to-be-Forgotten — Nullify Strategy

Customer `cust_003` (Carol Williams) has requested full data erasure.
The **nullify** strategy sets all PII fields to `NULL`.

In [ ]:
nullified = proc.forget(df, "customer_id", ["cust_003"], erasure_strategy="nullify")
nullified

Notice:
- `cust_003`'s email, name, phone, DOB, and address are now `null`
- `loyalty_tier` and `total_spend` are **preserved** (they're not marked `pii: true`)
- All other customers are **untouched**

## 2. Right-to-be-Forgotten — Hash Strategy

The **hash** strategy replaces PII with a SHA-256 one-way hash.
This **preserves referential integrity** — the same email hashes to the same value across tables.

In [ ]:
hashed = proc.forget(
    df, "customer_id", ["cust_003"],
    erasure_strategy="hash",
    hash_salt="lakelogic-gdpr-2024",
)
hashed

## 3. Right-to-be-Forgotten — Redact Strategy

The **redact** strategy replaces PII with a visible `***REDACTED***` marker.

In [ ]:
redacted = proc.forget(df, "customer_id", ["cust_003"], erasure_strategy="redact")
redacted

## 4. Batch Erasure — Multiple Subjects

Process a batch of GDPR requests at once.

In [ ]:
batch_ids = ["cust_001", "cust_004", "cust_005"]
batch_result = proc.forget(df, "customer_id", batch_ids)

print(f"Erased {len(batch_ids)} subjects")
print(f"Non-PII preserved: {batch_result['total_spend'].to_list()}")
batch_result

## 5. PII Masking — Anonymise Entire Datasets

Mask **all rows** (not just specific subjects) to create safe dev/test datasets.

In [ ]:
# Nullify all PII
masked_null = proc.mask_pii(df, strategy="nullify")
masked_null

In [ ]:
# Hash all PII — preserves referential integrity for JOINs
masked_hash = proc.mask_pii(df, strategy="hash", hash_salt="dev-env-2024")
masked_hash

In [ ]:
# Selective masking — only mask specific columns
selective = proc.mask_pii(df, columns=["email", "phone"])
selective

## 6. Hash Referential Integrity Check

Verify that the same value always hashes to the same output — critical for JOINs.

In [ ]:
dup_df = pl.DataFrame({
    "customer_id": ["c1", "c2", "c1"],
    "email": ["alice@example.com", "bob@example.com", "alice@example.com"],
    "full_name": ["Alice", "Bob", "Alice"],
    "phone": ["111", "222", "111"],
    "date_of_birth": ["1985-01-01", "1990-01-01", "1985-01-01"],
    "shipping_address": ["Addr1", "Addr2", "Addr1"],
})

hashed_dup = proc.mask_pii(dup_df, strategy="hash", hash_salt="dev-env-2024")

print(f"email[0] == email[2] (same person): {hashed_dup['email'][0] == hashed_dup['email'][2]}")
print(f"email[0] == email[1] (diff person): {hashed_dup['email'][0] == hashed_dup['email'][1]}")
hashed_dup

## 7. GDPR Audit Report

Generate a compliance-ready report for your records.

In [ ]:
report = generate_erasure_report(
    proc.contract,
    subject_column="customer_id",
    subject_ids=["cust_003"],
    erasure_strategy="nullify",
    affected_rows=1,
)

for key, value in report.items():
    print(f"  {key}: {value}")